# Phase 3: Pretrained Genomic Embedding Experiments

In this phase, we use a pretrained foundation model (`InstaDeepAI/nucleotide-transformer-50m-1000g`) to extract dense vector representations of our DNA sequences. We will then train classical interpretable models (Logistic Regression, Random Forest, XGBoost) on these embeddings and compare their performance against the Phase 1 $k$-mer baseline and Phase 2 Shallow CNN.

In [ ]:
!pip install transformers torch scikit-learn xgboost pandas numpy tqdm matplotlib seaborn

### 1. Import Libraries

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import torch
from pathlib import Path
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

# Add src to path
sys.path.append(os.path.abspath("../src"))
from features.embeddings import PretrainedEmbedder
from models.registry import ModelRegistry

### 2. Load Dataset

In [ ]:
data_dir = Path("../data")

# Load sequences and labels from processed CSVs
train_df = pd.read_csv(data_dir / "processed/train_data.csv")
test_df = pd.read_csv(data_dir / "processed/test_data.csv")

print(f"Train size: {len(train_df)}")
print(f"Test size: {len(test_df)}")

train_seqs = train_df['sequence'].tolist()
test_seqs = test_df['sequence'].tolist()
y_train = train_df['label'].values
y_test = test_df['label'].values

### 3. Extract Pretrained Embeddings
We use the `PretrainedEmbedder` from `src.features.embeddings` to generate mean-pooled embeddings for our sequences. This process runs inference via HuggingFace's transformers library.

In [ ]:
# Initialize Embedder
embedder = PretrainedEmbedder(model_name="InstaDeepAI/nucleotide-transformer-50m-1000g")

# Extract features
X_train_emb = embedder.get_embeddings(train_seqs, batch_size=32)
X_test_emb = embedder.get_embeddings(test_seqs, batch_size=32)

print(f"Train Embeddings Shape: {X_train_emb.shape}")
print(f"Test Embeddings Shape: {X_test_emb.shape}")

### 4. Train Classifiers on Embeddings
We train Logistic Regression, Random Forest, and XGBoost on the extracted dense representations.

In [ ]:
models = {
    "Logistic Regression": ModelRegistry.get_model("logistic_regression"),
    "Random Forest": ModelRegistry.get_model("random_forest", params={"n_estimators": 100}),
    "XGBoost": ModelRegistry.get_model("xgboost", params={"n_estimators": 100, "max_depth": 6, "learning_rate": 0.1})
}

results = []

for name, model in models.items():
    print(f"\nTraining {name}...")
    model.train(X_train_emb, y_train)
    
    # Evaluate
    preds = model.predict(X_test_emb)
    probs = model.predict_proba(X_test_emb)
    
    auroc = roc_auc_score(y_test, probs)
    auprc = average_precision_score(y_test, probs)
    f1 = f1_score(y_test, preds)
    
    results.append({
        "Model": name,
        "AUROC": auroc,
        "AUPRC": auprc,
        "F1-Score": f1
    })
    
    print(f"{name} Results - AUROC: {auroc:.4f} | AUPRC: {auprc:.4f} | F1: {f1:.4f}")

### 5. Compare with Phase 1 Baseline
*(Review Phase 1 output to see if pretrained embeddings provide the expected $\ge 0.02$ AUROC improvement as outlined in the decision gate!)*

In [ ]:
results_df = pd.DataFrame(results)
display(results_df)